In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

# 1. Models
from sklearn.linear_model import LogisticRegression              # Supervised
from sklearn.semi_supervised import LabelPropagation            # Semi-Supervised
from sklearn.ensemble import RandomForestClassifier             # Ensemble

# 2. Evaluation Metrics
from sklearn.metrics import accuracy_score, classification_report

# ==========================================
# STEP 1: LOAD YOUR DATASET
# ==========================================
# Replace 'your_dataset.csv' with the actual filename given in your practical exam
df = pd.read_csv('Customer-Churn.csv')

# Quick preview of your data
print("\nFirst 5 rows of the dataset:")
print(df.head())

# ==========================================
# STEP 2: PREPROCESSING
# ==========================================
# Define your target column name (Change 'target' to whatever column name is in your exam dataset)
target_column = 'Churn' 

X = df.drop(columns=[target_column])
y = df[target_column]

# Handle categorical variables if any (convert text columns to numbers)
X = pd.get_dummies(X, drop_first=True)

# Handle missing values and scale features
imputer = SimpleImputer(strategy='median')
X_imputed = imputer.fit_transform(X)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_imputed)

# Split data into Training and Testing sets (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)


# ==========================================
# STEP 3: SUPERVISED LEARNING MODEL
# ==========================================
print("\n--- Training Supervised Model (Logistic Regression) ---")
sup_model = LogisticRegression(random_state=42)
sup_model.fit(X_train, y_train)

sup_pred = sup_model.predict(X_test)
sup_acc = accuracy_score(y_test, sup_pred)
print(f"Supervised Accuracy: {sup_acc:.4f}")


# ==========================================
# STEP 4: SEMI-SUPERVISED LEARNING MODEL
# ==========================================
print("\n--- Training Semi-Supervised Model (Label Propagation) ---")
# To simulate semi-supervised learning, we hide (mark as -1) 50% of the training labels
y_train_semi = y_train.copy()
np.random.seed(42)
mask = np.random.rand(len(y_train_semi)) < 0.5  # 50% masked
y_train_semi_array = np.array(y_train_semi, dtype=float)
y_train_semi_array[mask] = -1  # -1 indicates unlabeled data

semi_model = LabelPropagation(kernel='rbf', gamma=20)
semi_model.fit(X_train, y_train_semi_array)

semi_pred = semi_model.predict(X_test)
semi_acc = accuracy_score(y_test, semi_pred)
print(f"Semi-Supervised Accuracy (with 50% unlabeled data): {semi_acc:.4f}")


# ==========================================
# STEP 5: ENSEMBLE LEARNING MODEL
# ==========================================
print("\n--- Training Ensemble Model (Random Forest) ---")
ensemble_model = RandomForestClassifier(n_estimators=100, random_state=42)
ensemble_model.fit(X_train, y_train)

ensemble_pred = ensemble_model.predict(X_test)
ensemble_acc = accuracy_score(y_test, ensemble_pred)
print(f"Ensemble Accuracy: {ensemble_acc:.4f}")


# ==========================================
# STEP 6: PERFORMANCE COMPARISON SUMMARY
# ==========================================
print("\n==========================================")
print("           PERFORMANCE SUMMARY            ")
print("==========================================")
print(f"1. Supervised (Logistic Regression): {sup_acc:.4f}")
print(f"2. Semi-Supervised (Label Prop):     {semi_acc:.4f}")
print(f"3. Ensemble (Random Forest):         {ensemble_acc:.4f}")
print("==========================================")


First 5 rows of the dataset:
   customerID  gender  SeniorCitizen Partner Dependents  tenure PhoneService  \
0  7590-VHVEG  Female              0     Yes         No       1           No   
1  5575-GNVDE    Male              0      No         No      34          Yes   
2  3668-QPYBK    Male              0      No         No       2          Yes   
3  7795-CFOCW    Male              0      No         No      45           No   
4  9237-HQITU  Female              0      No         No       2          Yes   

      MultipleLines InternetService OnlineSecurity  ... DeviceProtection  \
0  No phone service             DSL             No  ...               No   
1                No             DSL            Yes  ...              Yes   
2                No             DSL            Yes  ...               No   
3  No phone service             DSL            Yes  ...              Yes   
4                No     Fiber optic             No  ...               No   

  TechSupport StreamingTV Stream

ValueError: could not convert string to float: 'No'